# 06. Labeled training size

Owner: Jalil-g. Issue #6. Depends on baseline #3 / PR #21.

## Goal

Measure how labeled training size changes the fixed C1 baseline. Use nested balanced subsets and the same held-out rows throughout.

## Setup

No tuning. Subset sizes and the 2,000-row repeat were fixed before measurement. One thread, seed 42.

In [ ]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython import get_ipython
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

get_ipython().run_line_magic("run", "./02_split_metrics.ipynb")
NAME = "06_learning_curve"
COMMAND = "uv run --locked jupyter nbconvert --execute --to notebook --inplace notebooks/06_learning_curve.ipynb"
SIZES = [500, 1000, 2000, 5000, 8000]
REPEAT_SIZE = 2000
CONFIG = {
    "sizes": SIZES, "repeat_size": REPEAT_SIZE, "seed": 42, "threads": 1,
    "subsets": "class-wise seeded permutation of manifest-order training IDs; concatenate class prefixes; sort selected IDs",
    "vectorizer": {"analyzer": "word", "ngram_range": [1, 2], "lowercase": True,
                   "min_df": 2, "max_features": 50000, "sublinear_tf": True,
                   "norm": "l2", "dtype": "float32", "stop_words": None,
                   "token_pattern": r"(?u)\b\w\w+\b", "strip_accents": None},
    "classifier": {"C": 1.0, "solver": "liblinear", "max_iter": 2000,
                   "random_state": 42, "penalty": "l2", "tol": 1e-4,
                   "fit_intercept": True, "class_weight": None, "intercept_scaling": 1.0},
    "prediction_tolerance": 1e-12,
    "extra_versions": {name: importlib.metadata.version(name) for name in ["joblib", "matplotlib", "threadpoolctl"]},
}
source = json.loads((ROOT / "notebooks/06_learning_curve.ipynb").read_text())
CONFIG["notebook_source_sha256"] = value_hash(["".join(cell["source"]) for cell in source["cells"]])
START = time.perf_counter()

## Checks

Require a successful baseline with matching data, split, settings, and artifact hashes. Preserve the baseline confirmation locally. IDs only choose rows, never model features.

In [ ]:
parts = load_partitions()
train, validation = parts["train"], parts["validation"]
require(len(train) == 8000 and len(validation) == 2000, "Wrong partition sizes")
pointer_path = ROOT / "artifacts/03_word_baseline/latest.json"
require(pointer_path.exists(), "Run baseline notebook 03 first")
pointer = json.loads(pointer_path.read_text())
baseline_path = ROOT / pointer["result"]
require(file_hash(baseline_path) == pointer["result_sha256"], "Baseline result changed")
baseline = json.loads(baseline_path.read_text())
require(baseline["status"] == "completed" and baseline["fit_rows"] == 8000 and baseline["eval_rows"] == 2000,
        "Baseline did not complete the required split")
require(baseline["split_sha256"] == parts["split"]["split_sha256"] and
        baseline["data_sha256"] == parts["split"]["data_sha256"], "Baseline inputs differ")
for key in ["vectorizer", "classifier", "threads", "prediction_tolerance"]:
    require(baseline["config"][key] == CONFIG[key], "Baseline settings differ")
for path, expected in baseline["artifacts"].items():
    require(file_hash(ROOT / path) == expected, "Baseline artifact changed")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
private = ROOT / "artifacts" / NAME / run_id
private.mkdir(parents=True, exist_ok=False)
baseline_copy = private / "baseline_confirmation.json"
save_new_json(baseline_copy, baseline)
CONFIG["baseline_confirmation"] = {"result_sha256": file_hash(baseline_path),
    "git_sha": baseline["git_sha"], "notebook_source_sha256": baseline["config"]["notebook_source_sha256"],
    "validation_auroc": baseline["metrics"]["validation_auroc"]}

## Method

Shuffle each class once inside the training partition. Take equal prefixes for each size, then sort the selected IDs. This makes subsets nested. Fit a fresh TF-IDF vocabulary, IDF, and classifier for every arm.

Training AUROC uses that arm's fitted rows. Validation AUROC always uses the same 2,000 rows. The repeat checks numerical reproducibility, not uncertainty across seeds.

In [ ]:
def nested_subsets(frame, sizes, seed=42):
    rng = np.random.default_rng(seed)
    pools = {label: rng.permutation(frame.loc[frame.label == label, "id"].to_numpy()).tolist() for label in [0, 1]}
    indexed = frame.set_index("id", drop=False)
    subsets = {}
    for size in sizes:
        require(size % 2 == 0 and all(len(pool) >= size // 2 for pool in pools.values()), "Invalid subset size")
        selected = sorted(pools[0][:size // 2] + pools[1][:size // 2])
        subsets[size] = indexed.loc[selected].reset_index(drop=True)
    return subsets

subsets = nested_subsets(train, SIZES, SEED)
previous = set()
for size, frame in subsets.items():
    require(len(frame) == size and frame.label.value_counts().to_dict() == {0: size // 2, 1: size // 2}, "Subset balance changed")
    require(previous <= set(frame.id) <= set(train.id), "Subsets are not nested training rows")
    require(not set(frame.id) & set(validation.id), "Validation leakage")
    previous = set(frame.id)
subset_ids = {str(size): frame.id.tolist() for size, frame in subsets.items()}
subset_path = private / "subset_ids.json"
save_new_json(subset_path, subset_ids)
CONFIG["subset_id_sha256"] = {str(size): value_hash(frame.id.tolist()) for size, frame in subsets.items()}
CONFIG["validation_id_sha256"] = value_hash(validation.id.tolist())

def make_curve_model():
    settings = dict(CONFIG["vectorizer"])
    settings["ngram_range"] = tuple(settings["ngram_range"])
    settings["dtype"] = np.float32
    return Pipeline([("tfidf", TfidfVectorizer(**settings)),
                     ("logistic", LogisticRegression(**CONFIG["classifier"]))])

def fit_arm(frame, label):
    model = make_curve_model()
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        start = time.perf_counter()
        model.fit(frame.text, frame.label)
        fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        training_p = class1_probabilities(model, frame.text)
        validation_p = class1_probabilities(model, validation.text)
        train_metrics = evaluate_predictions(frame, pd.DataFrame({"id": frame.id, "p_ai": training_p}))
        val_metrics = evaluate_predictions(validation, pd.DataFrame({"id": validation.id, "p_ai": validation_p}))
        eval_seconds = time.perf_counter() - start
    prediction_path = private / f"{label}_validation.csv"
    pd.DataFrame({"id": validation.id, "p_ai": validation_p}).to_csv(prediction_path, index=False)
    return {"size": len(frame), "arm": label, "train_auroc": train_metrics["auroc"],
            "validation_auroc": val_metrics["auroc"], "fit_seconds": fit_seconds,
            "evaluation_seconds": eval_seconds, "vocabulary_size": len(model.named_steps["tfidf"].vocabulary_),
            "iterations": model.named_steps["logistic"].n_iter_.tolist()}, validation_p, prediction_path

## Run

In [ ]:
rows, predictions, artifacts = [], {}, [subset_path, baseline_copy]
try:
    for size in SIZES:
        row, values, path = fit_arm(subsets[size], str(size))
        rows.append(row)
        predictions[size] = values
        artifacts.append(path)
        # Keep finished arms if a later arm fails.
        save_new_json(private / f"{size}_aggregate.json", row)
    repeat, repeat_p, repeat_path = fit_arm(subsets[REPEAT_SIZE], f"{REPEAT_SIZE}_repeat")
    repeat_difference = float(np.max(np.abs(predictions[REPEAT_SIZE] - repeat_p)))
    require(repeat_difference <= CONFIG["prediction_tolerance"], "Repeat predictions changed")
    artifacts.append(repeat_path)
    baseline_predictions = pd.read_csv(ROOT / pointer["validation_predictions"], dtype={"id": "string"}, float_precision="round_trip")
    require(baseline_predictions.id.tolist() == validation.id.tolist(), "Baseline prediction order differs")
    baseline_difference = float(np.max(np.abs(predictions[8000] - baseline_predictions.p_ai.to_numpy())))
    require(baseline_difference <= CONFIG["prediction_tolerance"], "Full-size curve differs from baseline")
except Exception as exc:
    record_run(NAME, COMMAND, CONFIG, SEED, len(train), len(validation),
               {"validation_auroc": None, "completed_arms": rows}, time.perf_counter() - START,
               artifacts=artifacts, status="failed", error_type=type(exc).__name__)
    raise RuntimeError("Learning curve failed. Safe record saved; inspect private artifacts locally.") from None

## Results

In [ ]:
curve = pd.DataFrame(rows)
display(curve)
print(f"Repeat size: {REPEAT_SIZE}; maximum probability difference: {repeat_difference:.3g}")
print(f"8000-row baseline maximum probability difference: {baseline_difference:.3g}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve['size'], curve.train_auroc, 'o-', label='Training subset')
ax.plot(curve['size'], curve.validation_auroc, 'o-', label='Fixed validation')
ax.set(xlabel='Labeled training rows', ylabel='AUROC', ylim=(0.5, 1.01), title='Fixed C1 word baseline')
ax.set_xticks(SIZES)
ax.legend()
fig.tight_layout()
plot_path = ROOT / 'results' / NAME / f'{run_id}-curve.png'
plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_path, dpi=140)
plt.show()

## Takeaway

These are comparisons on one reused split. They do not measure uncertainty across data splits or predict Kaggle performance.

In [ ]:
first, last = rows[0], rows[-1]
gain = last['validation_auroc'] - first['validation_auroc']
gap = last['train_auroc'] - last['validation_auroc']
print(f"- Tried the fixed model at {SIZES}; no retuning.")
print(f"- Validation AUROC changed by {gain:+.6f} from 500 to 8000 rows. Full-size train/validation gap: {gap:.6f}.")
print("- Training scores above validation suggest overfitting. Use the measured curve only; more-data value beyond 8000 is unknown.")

## Save

Save all measured arms, including lower scores. Row predictions and subset membership stay private. The baseline PR still needs its own final review; this notebook does not close #3.

In [ ]:
metrics = {"validation_auroc": rows[-1]["validation_auroc"], "arms": rows,
           "repeat": repeat, "max_repeat_probability_difference": repeat_difference,
           "max_baseline_probability_difference": baseline_difference}
record_path = record_run(NAME, COMMAND, CONFIG, SEED, len(train), len(validation), metrics,
                         time.perf_counter() - START, artifacts=artifacts + [plot_path])
print("Saved measured curve, repeat check, timings, and hashes.")